# Pivot-order ablation (DAC paper)

Tests whether the cost-based pivot order explains PandoraZX's lower edge counts.
Every case runs PandoraZX `full_reduce` in two variants that differ **only** in the pivot order:

| variant | pivot order |
|---|---|
| `cost` | default: candidates sorted by $(\deg u-1)(\deg v-1)$, cheapest first |
| `id`   | the cost term removed from the `ORDER BY` of `zx_pivot_simp` and `zx_pivot_boundary_simp`, i.e. identifier order |

Each (case, variant) is repeated `REPEATS` times, interleaved, so the repeats also give the run-to-run variation the reviewers asked for.

**Before running:** use a *scratch* database (`build_pandora` drops its tables) and make sure no other benchmark is running, because runtimes are wall-clock times. The run cell appends to a CSV after every measurement and skips finished runs, so it can be interrupted and resumed.

In [ ]:
import asyncio
import json
import sys
from pathlib import Path

import asyncpg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "pandora").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the Pandora project root")


PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "benchmarking"))
import dac_ablation as dab       # run_case, CASES, case_name, id_ordered_pivot_sql
import dac_figures as figs       # paper plot style and colours

print(f"Project root: {PROJECT_ROOT}")

## Configuration

In [ ]:
DATABASE = "pandorazx_fig"      # scratch database, NOT the one used by zx_benchmarks.ipynb
REPEATS = 3
CASES = dab.CASES                 # {family: [circuit files]}; edit to add or remove sizes
RESULTS_PATH = PROJECT_ROOT / "benchmarking" / "dac_ablation.csv"
PLOT_PATH = PROJECT_ROOT / "benchmarking" / "zx_benchmark_plots" / "dac_ablation.pdf"

config = json.loads((PROJECT_ROOT / "default_config.json").read_text())
# Never run against the benchmark database: build_pandora drops its tables.
assert DATABASE != config.get("database", "postgres"), \
    f"DATABASE='{DATABASE}' is the benchmark database used by zx_benchmarks.ipynb; pick a scratch database"
config["database"] = DATABASE
for family, paths in CASES.items():
    print(f"{family:14s}", [dab.case_name(p) for p in paths])

In [ ]:
# Create the scratch database if it does not exist yet.
base = dict(config, database="postgres")
conn = await asyncpg.connect(host=base["host"], port=base["port"], user=base["user"],
                             password=base["password"], database=base["database"])
if not await conn.fetchval("SELECT 1 FROM pg_database WHERE datname = $1", DATABASE):
    await conn.execute(f'CREATE DATABASE "{DATABASE}"')
    print(f"Created database {DATABASE}")
else:
    print(f"Using existing database {DATABASE}")
await conn.close()

## Run (resumable)

In [ ]:
import csv

done = set()
if RESULTS_PATH.exists():
    previous = pd.read_csv(RESULTS_PATH)
    done = set(zip(previous.circuit, previous.variant, previous.run))

plan = [(family, path, run, variant)
        for family, paths in CASES.items() for path in paths
        for run in range(1, REPEATS + 1) for variant in ("cost", "id")]   # interleaved
todo = [p for p in plan if (dab.case_name(p[1]), p[3], p[2]) not in done]
print(f"{len(plan)} planned, {len(plan) - len(todo)} already done, {len(todo)} to run")

for i, (family, path, run, variant) in enumerate(todo, 1):
    row = {"family": family, "circuit": dab.case_name(path), "variant": variant, "run": run,
           **(await dab.run_case(config, path, variant))}
    new_file = not RESULTS_PATH.exists()
    with RESULTS_PATH.open("a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(row))
        if new_file:
            writer.writeheader()
        writer.writerow(row)
    print(f"[{i:3d}/{len(todo)}] {row['circuit']:14s} {variant:4s} run {run}: "
          f"{row['seconds']:7.2f} s, spiders {row['spiders_after']}, edges {row['edges_after']}")

## Results

In [ ]:
df = pd.read_csv(RESULTS_PATH)

# The rewriting is deterministic: every repeat of a (case, variant) must give the same diagram.
sizes = df.groupby(["circuit", "variant"])[["spiders_after", "edges_after"]].nunique()
assert (sizes == 1).all().all(), f"non-deterministic output:\n{sizes[(sizes > 1).any(axis=1)]}"

agg = (df.groupby(["family", "circuit", "gates", "variant"])
         .agg(runs=("run", "count"), time_mean=("seconds", "mean"), time_std=("seconds", "std"),
              spiders_before=("spiders_before", "first"), spiders=("spiders_after", "first"),
              edges_before=("edges_before", "first"), edges=("edges_after", "first"))
         .reset_index())
wide = agg.pivot_table(index=["family", "circuit", "gates", "spiders_before", "edges_before"],
                       columns="variant",
                       values=["runs", "time_mean", "time_std", "spiders", "edges"]).reset_index()
wide.columns = [f"{a}_{b}" if b else a for a, b in wide.columns]
wide = wide.sort_values(["family", "gates"]).reset_index(drop=True)
wide["edges_id_vs_cost_pct"] = 100 * (wide.edges_id / wide.edges_cost - 1)
wide["spiders_id_vs_cost_pct"] = 100 * (wide.spiders_id / wide.spiders_cost - 1)
wide["edge_change_cost_pct"] = 100 * (wide.edges_cost / wide.edges_before - 1)
wide["edge_change_id_pct"] = 100 * (wide.edges_id / wide.edges_before - 1)
wide["time_ratio_id_vs_cost"] = wide.time_mean_id / wide.time_mean_cost

display(wide[["family", "circuit", "gates",
              "edges_cost", "edges_id", "edges_id_vs_cost_pct",
              "spiders_cost", "spiders_id", "spiders_id_vs_cost_pct",
              "time_mean_cost", "time_std_cost", "time_mean_id", "time_std_id",
              "time_ratio_id_vs_cost", "runs_cost", "runs_id"]].round(2))

In [ ]:
# Figure in the paper's style: edge change relative to the input, cost vs identifier order.
from matplotlib.patches import Patch
from matplotlib.ticker import FixedLocator, FuncFormatter, NullLocator

figs.style()
families = list(dict.fromkeys(wide.family))
fig, axes = plt.subplots(1, len(families), figsize=(figs.COLUMN_WIDTH_IN, 1.6), sharey=True)
axes = np.atleast_1d(axes)
variant_style = {"cost": dict(facecolor=figs.COLOR["pandora"], edgecolor=figs.STROKE["pandora"]),
                 "id": dict(facecolor="#cdbfe6", edgecolor=figs.STROKE["pandora"], hatch="////")}
width = 0.38
for ax, family in zip(axes, families):
    d = wide[wide.family == family].reset_index(drop=True)
    for i, variant in enumerate(("cost", "id")):
        ratio = d[f"edges_{variant}"] / d.edges_before
        ax.bar(np.arange(len(d)) + (i - 0.5) * width, ratio - 1, bottom=1, width=width,
               linewidth=0.4, **variant_style[variant])
    ax.axhline(1, color=figs.MUTED, linewidth=0.7)
    ax.set_yscale("log")
    ax.set_xticks(range(len(d)), [figs.gates_label(g) for g in d.gates], fontsize=6)
    ax.tick_params(axis="x", length=0)
    ax.set_title(family, fontsize=7.5)
    ax.grid(True, axis="y", color=figs.GRID, linewidth=0.5)
    ax.set_axisbelow(True)
ticks = [1 + p / 100 for p in (-80, -50, 0, 100, 500)]
axes[0].yaxis.set_major_locator(FixedLocator(ticks))
axes[0].yaxis.set_major_formatter(FuncFormatter(figs.pct_from_ratio))
axes[0].yaxis.set_minor_locator(NullLocator())
axes[0].set_ylabel("Edge change (%)")
fig.legend(handles=[Patch(**variant_style["cost"], label="cost order (default)"),
                    Patch(**variant_style["id"], label="identifier order")],
           loc="upper center", ncol=2, frameon=False, bbox_to_anchor=(0.5, 1.08))
fig.tight_layout(rect=(0, 0, 1, 0.9))
PLOT_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(PLOT_PATH, bbox_inches="tight")
plt.show()
print(f"Saved {PLOT_PATH}")

## Text for the paper

A LaTeX table and a draft sentence for the "Why PandoraZX retains fewer edges" paragraph, filled in from the results above. Check the wording before pasting.

In [ ]:
def latex_circuit(name: str) -> str:
    if name.startswith("Adder"):
        return f"Adder, {name.removeprefix('Adder')} bit"
    if name.startswith("gf2^E"):
        n = name.removeprefix("gf2^E").removesuffix("_mult")
        return f"GF$(2^{{{n}}})$"
    return name.replace("IQP ", "IQP, ")


def k(x):
    return f"{x / 1000:.1f}"


def time_cell(mean, std):
    return f"{mean:.1f}" if np.isnan(std) else f"{mean:.1f}$\\pm${std:.1f}"


runs = int(wide[["runs_cost", "runs_id"]].min().min())
lines = [
    r"\begin{table}[t]",
    r"\centering",
    r"\caption{Pivot-order ablation: \PZX{} with cost-ordered (cost) and identifier-ordered (id) pivots;"
    rf" all other steps are identical. Counts refer to the output diagram, in thousands; times are the mean"
    rf" $\pm$ standard deviation over {runs} runs.}}",
    r"\label{tab:ablation}",
    r"\footnotesize",
    r"\setlength{\tabcolsep}{2.6pt}",
    r"\begin{tabular}{@{}lrrrrrr@{}}",
    r"\toprule",
    r" & \multicolumn{2}{c}{Edges (k)} & \multicolumn{2}{c}{Spiders (k)} & \multicolumn{2}{c}{Time (s)}\\",
    r"\cmidrule(lr){2-3}\cmidrule(lr){4-5}\cmidrule(l){6-7}",
    r"Circuit & cost & id & cost & id & cost & id \\",
    r"\midrule",
]
for r in wide.itertuples():
    lines.append(f"{latex_circuit(r.circuit)} & {k(r.edges_cost)} & {k(r.edges_id)} & "
                 f"{k(r.spiders_cost)} & {k(r.spiders_id)} & "
                 f"{time_cell(r.time_mean_cost, r.time_std_cost)} & {time_cell(r.time_mean_id, r.time_std_id)} \\\\")
lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
print("\n".join(lines))

In [ ]:
def span(series, fmt="{:+.0f}\\%"):
    lo, hi = series.min(), series.max()
    return fmt.format(lo) if np.isclose(lo, hi) else f"{fmt.format(lo)} to {fmt.format(hi)}"

FAMILY_TEXT = {"Binary adder": "binary adders", "GF multiplier": "GF multipliers", "IQP": "IQP circuits"}
per_family = "; ".join(
    f"{FAMILY_TEXT.get(family, family)}: {span(g.edges_id_vs_cost_pct)}"
    for family, g in wide.groupby("family", sort=False))
cv = 100 * (df.groupby(["circuit", "variant"]).seconds.std() / df.groupby(["circuit", "variant"]).seconds.mean())

print(
    "To isolate the effect of the order, we ran \\PZX{} with pivots applied in identifier order instead, "
    "keeping all other steps unchanged (Table~\\ref{tab:ablation}). "
    f"Identifier order changes the number of output edges by {span(wide.edges_id_vs_cost_pct)} "
    f"({per_family}) and the number of spiders by {span(wide.spiders_id_vs_cost_pct)}, "
    f"while the runtime changes by a factor of {wide.time_ratio_id_vs_cost.min():.2f}--{wide.time_ratio_id_vs_cost.max():.2f}. "
    f"Across {runs} repeated runs, the runtime varies by at most {np.nanmax(cv):.0f}\\% (coefficient of variation)."
)